<center>
    <p style="text-align:center">
    <img alt="arize logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="300"/>
        <br>
        <a href="https://docs.arize.com/arize/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/client_python">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-11t1vbu4x-xkBIHmOREQnYnYDH1GDfCg">Slack Community</a>
    </p>
</center>

# Trace-level evaluation: beyond input/output checks

*Worked example: a movie recommendation agent.*

## Why input/output checks aren't enough

The most common way to evaluate an LLM application is **end-to-end**: take the user's request, take the final answer, and judge whether the answer is good. That works for a single model call. It breaks down for **agents**, because an agent isn't one call — it's a *sequence of decisions*: which tools to call, in what order, with what arguments, and how to reason over the results.

End-to-end eval only inspects the two **endpoints** — the input and the final output. The interesting failures happen in the middle, where you can't see them:

- **Reasoning** — did the agent reason coherently toward the goal, or get there by luck?
- **Tool selection** — did it pick the right tools (and *skip* the wrong ones)?
- **Decision path** — did it call those tools in a sensible order, with sensible arguments?

The reason endpoints lie cuts both ways. A **correct-looking answer can come from a broken path** — the right result for the wrong reasons, which will fail the next time the inputs shift. And when an answer *is* wrong, the endpoints tell you *that* it's wrong but not *why* — the root cause is invisible unless you look inside the trace.

**The method:** capture the full trace, reconstruct the agent's *decision path* from its spans, and evaluate the intermediate steps — not just `input -> output`.

We'll demonstrate each idea on a movie recommendation agent. In this notebook, you will:

- Build and capture interactions (traces) from a movie recommendation agent
- Reconstruct the agent's **decision path** (the ordered tool calls and their arguments) from its spans
- Run **three evaluators**: an *endpoint* check (recommendation relevance) plus two *intermediate* checks — decision path (tool selection, order, and arguments) and reasoning (is the answer grounded in the tools' results?)
- See each evaluator catch a failure the endpoint check misses
- Format the evaluation outputs to match Arize AX's evaluation schema and log them back to the platform

✅ You will need a free Arize AX account and an OpenAI API key to run this notebook.

# Set Up Keys & Dependencies

In [ ]:
%pip install -qqqqq arize arize-otel "arize-phoenix-evals>=3" openinference-instrumentation-openai openinference-instrumentation-openai-agents openinference-instrumentation openai openai-agents nest_asyncio

In [ ]:
import os
from getpass import getpass

os.environ["ARIZE_SPACE_ID"] = globals().get("ARIZE_SPACE_ID") or getpass("🔑 Enter your Arize Space ID: ")

os.environ["ARIZE_API_KEY"] = globals().get("ARIZE_API_KEY") or getpass("🔑 Enter your Arize API Key: ")

os.environ["OPENAI_API_KEY"] = globals().get("OPENAI_API_KEY") or getpass("🔑 Enter your OpenAI API Key: ")

# Configure Tracing

In [ ]:
from arize.otel import register

model_id = "movie-recommendation-agent"
MODEL = "gpt-5.4-mini"
JUDGE_MODEL = "gpt-4.1-mini"
tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name=model_id,
    set_global_tracer_provider=True,
)

from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

# Build Movie Recommendation System

First, we need to define the tools that our recommendation system will use. For this example, we will define 3 tools:

1. Movie Selector: Based on the desired genre indicated by the user, choose up to 5 recent movies available for streaming
2. Reviewer: Find reviews for a movie. If given a list of movies, sort movies in order of highest to lowest ratings.
3. Preview Summarizer: For each movie, return a 1-2 sentence description

Our most ideal flow involves a user simply giving the system a type of movie they are looking for, and in return, the user gets a list of options returned with descriptions and reviews.

Each tool call the agent makes is a **decision point** — and every one of them is recorded as a span in the trace. That sequence of spans *is* the decision path we'll reconstruct and evaluate later, the part end-to-end eval never sees.

Let's test our agent and view the traces in Arize AX.

In [ ]:
import ast
from typing import List, Union

from agents import Agent, Runner, function_tool
from openai import OpenAI
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

client = OpenAI()


@function_tool
def movie_selector_llm(genre: str) -> List[str]:
    prompt = (
        f"List up to 5 recent popular streaming movies in the {genre} genre. "
        "Provide only movie titles as a Python list of strings."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    content = response.choices[0].message.content
    try:
        movie_list = ast.literal_eval(content)
        if isinstance(movie_list, list):
            return movie_list[:5]
    except Exception:
        return content.split("\n")


@function_tool
def reviewer_llm(movies: Union[str, List[str]]) -> str:
    if isinstance(movies, list):
        movies_str = ", ".join(movies)
        prompt = f"Sort the following movies by rating from highest to lowest and provide a short review for each:\n{movies_str}"
    else:
        prompt = f"Provide a short review and rating for the movie: {movies}"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


@function_tool
def preview_summarizer_llm(movie: str) -> str:
    prompt = f"Write a 1-2 sentence summary describing the movie '{movie}'."
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

In [ ]:
agent = Agent(
    name="MovieRecommendationAgentLLM",
    model=MODEL,
    tools=[movie_selector_llm, reviewer_llm, preview_summarizer_llm],
    instructions=(
        "You are a helpful movie recommendation assistant with access to three tools:\n"
        "1. MovieSelector: Given a genre, returns up to 5 recent streaming movies.\n"
        "2. Reviewer: Given one or more movie titles, returns reviews and sorts them by rating.\n"
        "3. PreviewSummarizer: Given a movie title, returns a 1-2 sentence summary.\n\n"
        "Your goal is to provide a helpful, user-friendly response combining relevant information."
    ),
)


async def main():
    user_input = "Which comedy movie should I watch?"
    result = await Runner.run(agent, user_input)
    print(result.final_output)


await main()

Next, we'll run the agent a few more times to generate additional traces. Feel free to adapt or customize the questions as you see fit.

In [ ]:
import time

questions = [
    "Which Batman movie should I watch?",
    "I want to watch a good romcom",
    "What is a very scary horror movie?",
    "Name a feel-good holiday movie",
    "Recommend a musical with great songs",
    "Give me a classic drama from the 90s",
]

for question in questions:
    result = await Runner.run(agent, question)

# Spans are exported in the background, so force a flush (and give Arize AX a
# moment to ingest) before we query them in the next section.
tracer_provider.force_flush()
time.sleep(5)

# Get Span Data from Arize AX

Before running our evaluations, we retrieve the span data from Arize AX and assemble what each evaluator needs.

We start with the **endpoints** — the user's question (`input`) and the agent's *final answer* (`output`). We take the final answer **only**, not a concatenation of every span's output: folding in tool outputs would blur the line between "what the user saw" and "what happened inside the trace," and quietly weaken the endpoint-vs-trace contrast this notebook is about.

> **Why read the LLM spans instead of the root span?** With the OpenAI Agents instrumentation, the root `AGENT` span doesn't record `input.value` / `output.value` — those attributes live on the underlying `LLM` spans. So we read the user's question from the first LLM call and the agent's final reply from its last text turn, using OpenInference's standard `llm.input_messages` / `llm.output_messages` attributes. (With an instrumentation that *does* populate the root span — many do — you could read `attributes.input.value` / `attributes.output.value` off the root spans directly.)

The endpoints are all that end-to-end eval ever sees. In the next step we reconstruct the intermediate signals it misses.

In [ ]:
from datetime import datetime, timedelta, timezone

from arize.client import ArizeClient

ax_client = ArizeClient(api_key=os.environ["ARIZE_API_KEY"])

primary_df = ax_client.spans.export_to_df(
    space_id=os.environ["ARIZE_SPACE_ID"],
    project_name=model_id,
    start_time=datetime.now(timezone.utc) - timedelta(days=7),
    end_time=datetime.now(timezone.utc),
)

In [ ]:
import json

import pandas as pd

SPAN_KIND = "attributes.openinference.span.kind"


def _as_list(value):
    """Span attributes can arrive as a JSON string or an already-parsed list."""
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return []
    return value if isinstance(value, list) else []


def user_query(input_messages):
    """The first user message in an LLM span's input."""
    for message in _as_list(input_messages):
        if message.get("message.role") == "user":
            contents = message.get("message.contents") or []
            return message.get("message.content") or "".join(
                c.get("message_content.text", "") for c in contents
            )
    return None


def assistant_text(output_messages):
    """The assistant's text reply in an LLM span (empty on tool-calling turns)."""
    texts = []
    for message in _as_list(output_messages):
        if message.get("message.role") == "assistant":
            if message.get("message.content"):
                texts.append(message["message.content"])
            for c in message.get("message.contents") or []:
                if c.get("message_content.type") == "text":
                    texts.append(c.get("message_content.text", ""))
    return " ".join(t for t in texts if t).strip() or None


# Restrict to the agent's own traces: each has a root span (no parent) of kind
# AGENT. This keeps the evaluators' own LLM calls — which are also traced into
# this project once you run the cells below — out of our evaluation set.
agent_roots = primary_df[(primary_df["parent_id"].isna()) & (primary_df[SPAN_KIND] == "AGENT")]
agent_trace_ids = agent_roots["context.trace_id"].unique()

llm_spans = primary_df[
    (primary_df[SPAN_KIND] == "LLM") & (primary_df["context.trace_id"].isin(agent_trace_ids))
].sort_values("start_time")

# Endpoint input  = the user's question (first user message in the trace).
# Endpoint output = the agent's FINAL answer only (its last text turn) — NOT a
# concatenation of every span's output, which would leak intermediate tool results
# into what is supposed to represent "what the user saw."
trace_df = pd.DataFrame(
    {
        "input": llm_spans.groupby("context.trace_id")["attributes.llm.input_messages"].apply(
            lambda s: next((q for q in s.map(user_query) if q), None)
        ),
        "output": llm_spans.groupby("context.trace_id")["attributes.llm.output_messages"].apply(
            lambda s: next((a for a in reversed(list(s.map(assistant_text))) if a), None)
        ),
    }
)
# Drop traces where we couldn't recover both endpoints, so no evaluator runs on a
# null input or output.
trace_df = trace_df.dropna(subset=["input", "output"])
trace_df.head()

## Reconstruct the intermediate signals

The `output` above is an endpoint — the agent's final answer. To evaluate the agent's *process*, we reconstruct two intermediate signals the endpoints never show, both from the trace's spans (`span kind == "TOOL"`, sorted by `start_time`):

- **`tool_path`** — the ordered tool calls the agent made, *with their arguments*. This is the decision path: tool selection, order, and the arguments each tool was called with.
- **`tool_io`** — what each tool was called *with* and what it *returned*. This is what lets us check whether the final answer is genuinely grounded in real tool results, rather than plausibly invented.

These two columns are exactly the signal end-to-end eval throws away.

In [ ]:
tool_spans = primary_df[primary_df[SPAN_KIND] == "TOOL"].sort_values("start_time")


# tool_path: the ordered tool calls WITH their arguments — the decision path,
# capturing tool selection, order, AND what each tool was asked to do.
def format_decision_path(group):
    return " -> ".join(
        f"{row['name']}({row['attributes.input.value']})" for _, row in group.iterrows()
    )


trace_df["tool_path"] = (
    tool_spans.groupby("context.trace_id")[["name", "attributes.input.value"]]
    .apply(format_decision_path)
    .reindex(trace_df.index)
    .fillna("No tools called")
)


# tool_io: each tool call's input AND output, so an evaluator can check whether the
# final answer is grounded in what the tools actually returned.
def format_tool_calls(group):
    lines = []
    for i, (_, row) in enumerate(group.iterrows(), start=1):
        lines.append(
            f"{i}. {row['name']} | input: {row['attributes.input.value']} "
            f"| output: {row['attributes.output.value']}"
        )
    return "\n".join(lines)


trace_df["tool_io"] = (
    tool_spans.groupby("context.trace_id")[
        ["name", "attributes.input.value", "attributes.output.value"]
    ]
    .apply(format_tool_calls)
    .reindex(trace_df.index)
    .fillna("No tools called")
)
trace_df[["input", "tool_path", "tool_io"]].head()

# Define and Run Evaluators

We'll now run **three evaluators** — one endpoint check and two intermediate checks, each reading a different column:

1. **Recommendation relevance — an *endpoint* check.** Reads `input` and the final `output` and asks whether the recommendations match the request. This is exactly what end-to-end eval does.
2. **Decision path — an *intermediate* check.** Reads `input` and `tool_path` (the ordered tool calls *with their arguments*) and asks whether the agent selected the right tools, in a sensible order, with sensible arguments. It never looks at the final output — it judges *how* the agent got there.
3. **Reasoning / support — an *intermediate* check.** Reads `input`, `tool_io`, and `output`, and asks whether the final answer is actually *grounded in the tool results* — catching answers that assert facts (a rating, a review, a title) that no tool produced.

The contrast is the point of this notebook: a trace can **pass the endpoint check and still fail an intermediate one** — a good-looking answer reached through a broken tool order, or one that invents a fact the tools never returned. All three run as LLM-as-a-judge classifiers.

In [ ]:
DECISION_PATH = """
You are evaluating an agent's DECISION PATH: the ordered tool calls it made to
answer a request — which tools, in what order, and with what arguments. You are
NOT judging the final answer text, and you are NOT judging whether each tool's
output was correct — only whether the agent's choices were sensible.

The agent has three tools available:
- movie_selector_llm(genre): returns candidate movies. Must come FIRST, because the
  other tools operate on the selected movies.
- reviewer_llm(movies): reviews and sorts movies. Only meaningful AFTER selection,
  and should be called with the movies that were actually selected.
- preview_summarizer_llm(movie): summarizes a movie. Only meaningful AFTER selection.

You will be given:
1. The user input that initiated the trace
2. The ordered tool calls the agent executed, with their arguments

##
User Input:
{input}

Decision Path (ordered tool calls with arguments):
{tool_path}
##

Respond with exactly one word: `correct` or `incorrect`.
1. `correct` ->
- movie_selector_llm is called before reviewer_llm or preview_summarizer_llm, AND
- each tool is called with sensible arguments (e.g. reviewer_llm receives the
  movies that were selected, not an empty or unrelated list).
2. `incorrect` ->
- a tool that operates on movies (reviewer_llm / preview_summarizer_llm) runs
  before any movies have been selected, the selection step is missing, OR a tool is
  called with nonsensical arguments (empty/placeholder inputs, or movies that were
  never selected).
"""

In [ ]:
RECOMMENDATION_RELEVANCE = """
You are evaluating the relevance of movie recommendations provided by an LLM application.

You will be given:
1. The user input that initiated the trace
2. The list of movie recommendations output by the system

##
User Input:
{input}

Recommendations:
{output}
##

Respond with exactly one word: `correct` or `incorrect`.
1. `correct` ->
- All recommended movies match the requested genre or criteria in the user input.
- The recommendations are relevant to the user's request and are not repetitive.
2. `incorrect` ->
- One or more recommendations do not match the requested genre or criteria, or the
  recommendations are repetitive.
"""

In [ ]:
REASONING_SUPPORT = """
You are checking whether an agent's FINAL ANSWER is SUPPORTED by the actual
results its tools returned.

You are NOT judging tool order (that's the decision-path check) or genre match
(that's the relevance check). Judge ONLY whether every concrete claim in the final
answer — titles, ratings, scores, review quotes, plot facts — is grounded in the
tool outputs below. An answer that asserts a fact no tool produced (for example a
specific rating) is unsupported, even if it sounds plausible.

##
User Input:
{input}

Tool calls and their results (in order):
{tool_io}

Final Answer:
{output}
##

Respond with exactly one word: `correct` or `incorrect`.
1. `correct` -> every concrete claim in the final answer is supported by the tool
   results above.
2. `incorrect` -> the final answer asserts at least one concrete fact (a rating,
   score, review, or title) that does not appear in the tool results.
"""

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator, async_evaluate_dataframe
from openinference.instrumentation import suppress_tracing
import nest_asyncio

nest_asyncio.apply()

judge_llm = LLM(provider="openai", model=JUDGE_MODEL)

# Each evaluator reads the columns named in its template and returns a Score
# (label + numeric score + explanation). The name becomes the eval name in Arize AX.
relevance_evaluator = ClassificationEvaluator(
    name="relevance",
    llm=judge_llm,
    prompt_template=RECOMMENDATION_RELEVANCE,
    choices={"correct": 1.0, "incorrect": 0.0},
)
path_evaluator = ClassificationEvaluator(
    name="decision_path",
    llm=judge_llm,
    prompt_template=DECISION_PATH,
    choices={"correct": 1.0, "incorrect": 0.0},
)
reasoning_evaluator = ClassificationEvaluator(
    name="reasoning",
    llm=judge_llm,
    prompt_template=REASONING_SUPPORT,
    choices={"correct": 1.0, "incorrect": 0.0},
)

EVALUATORS = [relevance_evaluator, path_evaluator, reasoning_evaluator]

# The judges are themselves LLM calls, which OpenAI auto-instrumentation would trace
# into this same project. suppress_tracing keeps those evaluator spans out, so the
# project stays just the agent's traces.
with suppress_tracing():
    results_df = await async_evaluate_dataframe(dataframe=trace_df, evaluators=EVALUATORS)

results_df.head()

## Seeing what each evaluator catches

On well-behaved traces the three evaluators agree. The point of trace-level evaluation is what happens when they *don't*. Below we run the three judges on three controlled cases that **mirror the real trace schema** (`input`, `output`, `tool_path`, `tool_io`) — each with a relevant-looking answer, so the **endpoint (relevance) check passes every time**:

- **Clean run** — selects, reviews, summarizes; the answer matches the tool results. Everything passes.
- **Broken order** — reviews a movie before any movie has been selected. The **decision path** check catches it; relevance and support cannot.
- **Unsupported claim** — good tool order, but the answer cites a *"9.2/10"* rating that appears in no tool output. The **reasoning / support** check catches it; relevance and decision path cannot.

Two intermediate failures, two different lenses — and both invisible to the endpoint check.

In [ ]:
# Controlled cases that mirror the real trace schema (input, output, tool_path,
# tool_io). Each answer looks relevant, so the endpoint check passes — the
# failures live in the intermediate signals. tool_path shows the calls + arguments;
# tool_io adds the results. Each transcript matches its path, broken-order included.
clean_path = (
    'movie_selector_llm({"genre":"comedy"}) -> '
    'reviewer_llm({"movies":["The Grand Budapest Hotel","Superbad"]}) -> '
    'preview_summarizer_llm({"movie":"Superbad"})'
)
clean_io = (
    '1. movie_selector_llm | input: {"genre":"comedy"} | output: ["The Grand Budapest Hotel", "Superbad"]\n'
    '2. reviewer_llm | input: {"movies":["The Grand Budapest Hotel","Superbad"]} | output: The Grand Budapest Hotel — witty and stylish; Superbad — raunchy coming-of-age comedy.\n'
    '3. preview_summarizer_llm | input: {"movie":"Superbad"} | output: Two friends navigate one wild night before graduation.'
)
broken_order_path = 'reviewer_llm({"movies":[]}) -> movie_selector_llm({"genre":"comedy"})'
broken_order_io = (
    '1. reviewer_llm | input: {"movies":[]} | output: No movies were provided to review.\n'
    '2. movie_selector_llm | input: {"genre":"comedy"} | output: ["The Grand Budapest Hotel", "Superbad"]'
)
unsupported_path = (
    'movie_selector_llm({"genre":"comedy"}) -> '
    'reviewer_llm({"movies":["Superbad"]}) -> '
    'preview_summarizer_llm({"movie":"Superbad"})'
)
no_rating_io = (
    '1. movie_selector_llm | input: {"genre":"comedy"} | output: ["The Grand Budapest Hotel", "Superbad"]\n'
    '2. reviewer_llm | input: {"movies":["Superbad"]} | output: Superbad — a funny, heartfelt coming-of-age comedy.\n'
    '3. preview_summarizer_llm | input: {"movie":"Superbad"} | output: Two friends navigate one wild night.'
)

demo_df = pd.DataFrame(
    {
        "case": ["clean run", "broken order", "unsupported claim"],
        "input": ["Which comedy movie should I watch?"] * 3,
        "output": [
            "Two great comedy picks: 'The Grand Budapest Hotel' and 'Superbad'.",
            "Two great comedy picks: 'The Grand Budapest Hotel' and 'Superbad'.",
            "Watch 'Superbad' — critics rated it 9.2/10.",  # no tool produced this rating
        ],
        "tool_path": [clean_path, broken_order_path, unsupported_path],
        "tool_io": [clean_io, broken_order_io, no_rating_io],
    }
)

with suppress_tracing():
    demo_results = await async_evaluate_dataframe(dataframe=demo_df, evaluators=EVALUATORS)


def label_of(cell):
    """Pull the classification label out of a Phoenix score cell (a dict)."""
    if isinstance(cell, str):
        try:
            cell = json.loads(cell)
        except json.JSONDecodeError:
            return None
    if isinstance(cell, dict):
        return cell.get("label")
    return getattr(cell, "label", None)


pd.DataFrame(
    {
        "case": demo_df["case"],
        "relevance": demo_results["relevance_score"].apply(label_of).values,
        "decision path": demo_results["decision_path_score"].apply(label_of).values,
        "reasoning": demo_results["reasoning_score"].apply(label_of).values,
    }
)

# Log Results Back to Arize AX

The final step is to log our results back to Arize AX. We attach each trace's three evaluations to its **root span**, using the `trace_eval.<name>.label/score/explanation` column convention that Arize AX expects. After running the cell below, you'll be able to view your trace-level evaluations on the platform, complete with labels, scores, and explanations.

In [ ]:
def _unpack(cell):
    """Phoenix returns each evaluator's result as a Score dict in `<name>_score`."""
    if isinstance(cell, str):
        try:
            cell = json.loads(cell)
        except json.JSONDecodeError:
            return None, None, None
    if isinstance(cell, dict):
        return cell.get("label"), cell.get("score"), cell.get("explanation")
    return getattr(cell, "label", None), getattr(cell, "score", None), getattr(cell, "explanation", None)


# Map each evaluator's score column to the Arize AX eval name shown in the UI.
EVAL_COLUMNS = {
    "RecommendationRelevance": "relevance_score",
    "DecisionPath": "decision_path_score",
    "ReasoningSupport": "reasoning_score",
}

# Build the trace_eval.* columns Arize AX expects (rows aligned positionally to trace_df).
eval_df = pd.DataFrame()
for ax_name, score_col in EVAL_COLUMNS.items():
    unpacked = results_df[score_col].apply(_unpack)
    eval_df[f"trace_eval.{ax_name}.label"] = unpacked.map(lambda t: t[0]).values
    eval_df[f"trace_eval.{ax_name}.score"] = unpacked.map(lambda t: t[1]).values
    eval_df[f"trace_eval.{ax_name}.explanation"] = unpacked.map(lambda t: t[2]).values
eval_df["context.trace_id"] = trace_df.index.values

# Map each trace to its root (AGENT) span so the evals attach to the root span.
root_spans = agent_roots[["context.trace_id", "context.span_id"]]
log_df = eval_df.merge(root_spans, on="context.trace_id", how="inner").set_index(
    "context.span_id", drop=False
)

# Reuse the ArizeClient from the export step; update_evaluations attaches each
# row's trace_eval.* columns to the span named by its context.span_id.
resp = ax_client.spans.update_evaluations(
    space_id=os.environ["ARIZE_SPACE_ID"],
    project_name=model_id,
    dataframe=log_df,
)

The three trace-level evaluations — `RecommendationRelevance`, `DecisionPath`, and `ReasoningSupport` — are now logged onto each real agent trace in Arize AX, each with the judge's label, score, and explanation.

![Trace Evals in Arize AX](https://storage.googleapis.com/arize-phoenix-assets/assets/images/trace-level-evals-2.png)

## Takeaway

We ran three evaluators over the same traces, each asking a different question and reading a different signal:

- **relevance** (endpoint) — *did the final answer look right?* Reads only `input` and `output` — exactly what end-to-end eval sees.
- **decision path** (intermediate) — *did the agent pick the right tools, in the right order, with sensible arguments?* Reads `tool_path`.
- **reasoning / support** (intermediate) — *is the final answer actually grounded in what the tools returned?* Reads `tool_io`.

The controlled cases show why all three matter: every answer looked relevant, yet the decision-path check caught a broken tool order and the support check caught a fabricated rating — two *different* failures, both invisible at the endpoints. That's the core lesson: **a good-looking answer can hide a broken process, and only intermediate evals — reading signals reconstructed from spans — can see it.**

The pattern generalizes. For any intermediate step you care about, reconstruct the relevant signal from spans into a column (`tool_path`, `tool_io`, retrieved context, a sub-agent's handoff, …), write a judge that reads that column, and run it alongside your endpoint eval.